# Task B.3 — Human Calibration với Cohen's Kappa

Compute Cohen's kappa between human labels (10 samples) and LLM judge.

**Inputs:**
- `human_labels.csv` (10 rows, manually labeled)
- `pairwise_results.csv` (LLM judge output, swap-and-average)


In [ ]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score

human_df = pd.read_csv('human_labels.csv')
judge_df = pd.read_csv('pairwise_results.csv').head(len(human_df))

def normalize(s):
    s = str(s).strip().lower()
    if s in ('a', 'answer_a'): return 'A'
    if s in ('b', 'answer_b'): return 'B'
    return 'tie'

human = [normalize(x) for x in human_df['human_winner']]
judge = [normalize(x) for x in judge_df['winner_after_swap']]

print(f'Human: {human}')
print(f'Judge: {judge}')
print(f'N: {len(human)}')


In [ ]:
kappa = cohen_kappa_score(human, judge)
print(f"Cohen's kappa: {kappa:.3f}")

if kappa < 0:
    print('WORSE than chance')
elif kappa < 0.2:
    print('Slight agreement')
elif kappa < 0.4:
    print('Fair agreement')
elif kappa < 0.6:
    print('Moderate agreement -- ok cho monitoring')
elif kappa < 0.8:
    print('Substantial agreement -- production ready')
else:
    print('Almost perfect')


## Disagreement details

Identify which pairs disagree and the human reasoning.

In [ ]:
disagreements = []
for i, (h, j) in enumerate(zip(human, judge)):
    if h != j:
        disagreements.append({
            'qid': int(human_df.iloc[i]['question_id']),
            'human': h, 'judge': j,
            'confidence': human_df.iloc[i]['confidence'],
            'notes': human_df.iloc[i].get('notes', '')
        })

import pandas as pd
pd.DataFrame(disagreements)


## Root cause hypothesis (kappa < 0.6)

Likely culprits:
1. **Length bias** — judge picks longer/more thorough answer; humans prefer concise/correct
2. **Surface accuracy vs. depth** — judge weighs factual coverage; humans weigh actionability
3. **Sample size = 10** is small for stable kappa; need 30+ for tight CI

See `judge_bias_report.md` for quantification of bias 1.